# Fase 4 · M02: Análisis del Target

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 4 — EDA Final |
| **Módulo** | M02 — Target |

---

## 🎯 Qué hace

Analiza la variable target `abandono`: distribución, desbalanceo de clases, evolución temporal, tasas por rama y titulación.

## 📋 Requisitos

- `data/04_eda/df_eda_final.parquet`

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `docs/html/fase4/m02_target.html` | Análisis del target abandono |

## 🔄 Flujo

```
data/04_eda/df_eda_final.parquet
    ↓ Distribución target
    ↓ Desbalanceo 70.8% / 29.2%
    ↓ Evolución por cohorte
    → docs/html/fase4/m02_target.html
```

## ➡️ Siguiente

`f4_m03_distribuciones_num.ipynb` — distribuciones de variables numéricas


In [1]:
# ── CSS + JS para botones Interpretar ────────────────────────────────────────
js_css_html = (
    '<style>'
    '.ibt{display:inline-flex;align-items:center;gap:5px;padding:5px 12px;'
    'background:#3182ce;color:#fff;border:none;border-radius:5px;'
    'font-size:12px;font-weight:600;cursor:pointer;float:right;margin-top:-2px;}'
    '.ibt:hover{background:#2b6cb0;}'
    '.ipn{display:none;background:#f7fafc;border:1px solid #e2e8f0;'
    'border-radius:6px;padding:14px 16px;margin-top:8px;font-size:13px;'
    'color:#2d3748;line-height:1.6;}'
    '.ipn.vis{display:block;}'
    '.tg{background:#c6f6d5;color:#276749;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '.tb{background:#fed7d7;color:#9b2c2c;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '.tm{background:#fefcbf;color:#744210;padding:1px 6px;border-radius:3px;font-weight:600;}'
    '</style>'
    '<script>'
    'function tg(id){'
    'var p=document.getElementById(id);'
    'var b=document.querySelector("[data-pid=\'" + id + "\']");'
    'if(!p||!b)return;'
    'var vis=p.classList.toggle("vis");'
    'b.textContent=vis?"✖ Cerrar":"📖 Interpretar";}'
    '</script>'
)

def h2btn(titulo, pid):
    return (
        f'<div style="display:flex;align-items:center;justify-content:space-between;'
        f'margin:28px 0 8px;">'
        f'<h2 style="font-size:18px;font-weight:700;color:#1a202c;margin:0;">{titulo}</h2>'
        f'<button class="ibt" data-pid="{pid}" onclick="tg(\'{pid}\')">' 
        f'📖 Interpretar</button></div>'
    )

def panel(pid, texto):
    return f'<div id="{pid}" class="ipn">{texto}</div>'


In [2]:
# ============================================================================
# CELDA 1: CONFIGURACION
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

ROOT = Path.cwd()
for _ in range(6):
    if (ROOT / 'src').exists():
        break
    ROOT = ROOT.parent


sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from src.config import (RUTA_HTML, ETIQUETAS_VARIABLES, DICCIONARIO_RAMAS,
                        info_entorno)
from src.utils import crear_directorios, formato_numero_es

# ── Paneles de interpretación dinámica ──────────────
panel_rama = panel('rama', 'Predictor de abandono estructural por área de conocimiento. Combinada con nota_1er_anio es más informativa. Variable incluida en Fase 5 y analizada en equidad (Fase 6 fairness).')
panel_sexo_beca = panel('sexo-beca', '<b>Sexo:</b> diferencia moderada — analizada en Fase 6 fairness. <b>Beca/apoyo económico:</b> más discriminativa — sin apoyo abandona más. Ambas analizadas con <span class=\"tg\">fairlearn</span> para medir Disparate Impact y Equal Opportunity.')
panel_via_acceso = panel('via-acceso', 'Captura el perfil de entrada al sistema universitario. Cardinalidad media — OrdinalEncoder para árboles, OneHotEncoder para lineales. Aporta contexto al modelo sobre el tipo de acceso.')

from src.html import (generar_kpis_html, generar_seccion_html,
                      generar_html_navegacion_completa, guardar_html)
from src.html.render import render_base_html

RUTA_EDA = ROOT / 'data' / '04_eda'
RUTA_FASE4_HTML = RUTA_HTML / 'fase4'
crear_directorios([RUTA_FASE4_HTML])

fmt = formato_numero_es
COLOR_ABANDONO = '#e53e3e'
COLOR_PERMANENCIA = '#3182ce'

def etiq(col):
    return ETIQUETAS_VARIABLES.get(col, col)

# Config Plotly global
plotly_cfg = {'displayModeBar': True, 'displaylogo': False,
              'modeBarButtonsToRemove': ['lasso2d', 'select2d']}

info_entorno()

✓ Directorios verificados: 1
✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓

In [3]:
# ============================================================================
# CELDA 2: CARGA Y METRICAS GLOBALES
# ============================================================================

df = pd.read_parquet(RUTA_EDA / 'df_eda_final.parquet')

n_filas = len(df)
tasa_global = df['abandono'].mean() * 100
n_abandonan = int(df['abandono'].sum())
n_permanecen = n_filas - n_abandonan

def tasa_por_grupo(col, min_n=0):
    grupo = df.groupby(col)['abandono'].agg(['mean', 'sum', 'count']).reset_index()
    grupo.columns = [col, 'tasa', 'n_abandono', 'n_total']
    grupo['tasa_pct'] = (grupo['tasa'] * 100).round(1)
    grupo['n_abandono'] = grupo['n_abandono'].astype(int)
    grupo['n_total'] = grupo['n_total'].astype(int)
    if min_n > 0:
        grupo = grupo[grupo['n_total'] >= min_n]
    return grupo.sort_values('tasa', ascending=False)

rama_g = tasa_por_grupo('rama')
rama_max, rama_max_tasa = rama_g.iloc[0]['rama'], rama_g.iloc[0]['tasa_pct']
sexo_g = tasa_por_grupo('sexo')
sexo_max, sexo_max_tasa = sexo_g.iloc[0]['sexo'], sexo_g.iloc[0]['tasa_pct']
via_g = tasa_por_grupo('via_acceso', min_n=30)
via_max, via_max_tasa = via_g.iloc[0]['via_acceso'], via_g.iloc[0]['tasa_pct']
tit_g = tasa_por_grupo('titulacion', min_n=30)
tit_max, tit_max_tasa = tit_g.iloc[0]['titulacion'], tit_g.iloc[0]['tasa_pct']
tit_max_corto = tit_max.replace('Grado en ', '')[:25]
sit_g = tasa_por_grupo('situacion_laboral', min_n=20)
sit_max, sit_max_tasa = sit_g.iloc[0]['situacion_laboral'], sit_g.iloc[0]['tasa_pct']

print(f'Tasa global: {tasa_global:.1f}%')
print(f'Rama max: {DICCIONARIO_RAMAS.get(rama_max, rama_max)} ({rama_max_tasa}%)')
print(f'Sexo max: {sexo_max} ({sexo_max_tasa}%)')
print(f'Tit max: {tit_max_corto} ({tit_max_tasa}%)')

# KPIs adicionales — usando n_anios_beca (tuvo_beca ya no existe en el dataset)
df['_tuvo_beca'] = (df['n_anios_beca'] > 0).astype(int)
beca_g = tasa_por_grupo('_tuvo_beca')
tasa_sin_beca = beca_g[beca_g['_tuvo_beca']==0]['tasa_pct'].values[0] if 0 in beca_g['_tuvo_beca'].values else 0
tasa_con_beca = beca_g[beca_g['_tuvo_beca']==1]['tasa_pct'].values[0] if 1 in beca_g['_tuvo_beca'].values else 0

import pandas as pd
bins_edad = [0, 18, 20, 22, 25, 30, 100]
labels_edad = ['18 o menos', '19-20', '21-22', '23-25', '26-30', '31+']
df['grupo_edad'] = pd.cut(df['edad_entrada'], bins=bins_edad, labels=labels_edad)
tasa_edad_joven = df[df['grupo_edad']=='18 o menos']['abandono'].mean() * 100
tasa_edad_mayor = df[df['grupo_edad']=='26-30']['abandono'].mean() * 100

n_titulaciones = df['titulacion'].nunique()
n_ramas = df['rama'].nunique()

rama_min = rama_g.iloc[-1]['rama']
rama_min_tasa = rama_g.iloc[-1]['tasa_pct']

print(f'Beca: Sin={tasa_sin_beca}% / Con={tasa_con_beca}%')
print(f'Edad: 18-={tasa_edad_joven:.1f}% / 26-30={tasa_edad_mayor:.1f}%')
print(f'Titulaciones: {n_titulaciones}, Ramas: {n_ramas}')

Tasa global: 29.2%
Rama max: Ingeniería y Arquitectura (37.4%)
Sexo max: Hombre (36.8%)
Tit max: Ingeniería Eléctrica (59.7%)
Beca: Sin=46.8% / Con=22.5%
Edad: 18-=21.5% / 26-30=46.9%
Titulaciones: 40, Ramas: 5


In [4]:
# ============================================================================
# CELDA 3: GRAFICOS INTERACTIVOS (PLOTLY)
# ============================================================================

graficos_html = {}
plotly_cfg = {'displayModeBar': True, 'displaylogo': False,
              'modeBarButtonsToRemove': ['lasso2d', 'select2d'],
              'responsive': True}

PALETA = ['#3182ce', '#e53e3e', '#38a169', '#ed8936', '#805ad5',
          '#d69e2e', '#319795', '#b83280', '#2c5282', '#c53030']

def hover_h():
    return '<b>%{y}</b><br>Tasa de abandono: %{x:.1f}%<br>Abandonos: %{customdata[0]:,}<br>Total: %{customdata[1]:,}<extra></extra>'

# ---------------------------------------------------------------
# 1. RAMA — Barras horizontales con barras dobles (abandono + permanencia)
# ---------------------------------------------------------------
rama_g['rama_nombre'] = rama_g['rama'].map(DICCIONARIO_RAMAS)
rama_g = rama_g.sort_values('tasa_pct', ascending=True)
rama_g['perm_pct'] = (100 - rama_g['tasa_pct']).round(1)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=rama_g['rama_nombre'], x=rama_g['tasa_pct'], orientation='h',
    name='Abandono', marker_color=COLOR_ABANDONO,
    text=[f"{t}%" for t in rama_g['tasa_pct']], textposition='inside',
    textfont=dict(color='white', size=12),
    hovertemplate='<b>%{y}</b><br>Abandono: %{x:.1f}%<br>n=%{customdata[0]:,} de %{customdata[1]:,}<extra></extra>',
    customdata=list(zip(rama_g['n_abandono'], rama_g['n_total']))
))
fig.add_trace(go.Bar(
    y=rama_g['rama_nombre'], x=rama_g['perm_pct'], orientation='h',
    name='Permanencia', marker_color=COLOR_PERMANENCIA, opacity=0.6,
    text=[f"{t}%" for t in rama_g['perm_pct']], textposition='inside',
    textfont=dict(color='white', size=11),
    hovertemplate='<b>%{y}</b><br>Permanencia: %{x:.1f}%<extra></extra>',
))
fig.update_layout(barmode='stack', title='Abandono por Rama de Conocimiento',
                  xaxis_title='Porcentaje (%)', height=350, template='plotly_white',
                  legend=dict(orientation='h', x=1, y=1.02, xanchor='right', yanchor='bottom', bgcolor='rgba(255,255,255,0.85)', bordercolor='#e2e8f0', borderwidth=1, font=dict(size=11)),
                  margin=dict(b=40))
graficos_html['rama'] = fig.to_html(full_html=False, include_plotlyjs=False, config=plotly_cfg)

# ---------------------------------------------------------------
# 2. SEXO — Donut comparativo lado a lado (subplot)
# ---------------------------------------------------------------
from plotly.subplots import make_subplots

sexo_g_sorted = sexo_g.sort_values('tasa_pct', ascending=False)
fig = make_subplots(rows=1, cols=2, specs=[[{'type':'pie'}, {'type':'pie'}]],
                    subplot_titles=[f'{s}' for s in sexo_g_sorted['sexo']])

for i, (_, row) in enumerate(sexo_g_sorted.iterrows()):
    fig.add_trace(go.Pie(
        values=[row['tasa_pct'], 100 - row['tasa_pct']],
        labels=['Abandono', 'Permanencia'],
        marker_colors=[COLOR_ABANDONO, COLOR_PERMANENCIA],
        hole=0.5, textinfo='percent', textfont_size=14,
        hovertemplate=f'<b>{row["sexo"]}</b><br>%{{label}}: %{{value:.1f}}%<br>n={int(row["n_total"]):,}<extra></extra>',
        name=row['sexo'],
    ), row=1, col=i+1)

fig.update_layout(title='Abandono por Sexo', height=320, template='plotly_white',
                  showlegend=True, legend=dict(orientation='h', y=-0.05, x=0.5, xanchor='center'),
                  margin=dict(t=60, b=40))
for i, (_, row) in enumerate(sexo_g_sorted.iterrows()):graficos_html['sexo'] = fig.to_html(full_html=False, include_plotlyjs=False, config=plotly_cfg)

# ---------------------------------------------------------------
# 3. BECA — Donut comparativo (usando n_anios_beca derivado en celda anterior)
# ---------------------------------------------------------------
beca_g = tasa_por_grupo('_tuvo_beca')
beca_labels = {0: 'Sin beca', 1: 'Con beca'}
beca_g['label'] = beca_g['_tuvo_beca'].map(beca_labels)
beca_g = beca_g.sort_values('tasa_pct', ascending=False)

fig = make_subplots(rows=1, cols=2, specs=[[{'type':'pie'}, {'type':'pie'}]],
                    subplot_titles=[f'{r["label"]}' for _, r in beca_g.iterrows()])
for i, (_, row) in enumerate(beca_g.iterrows()):
    fig.add_trace(go.Pie(
        values=[row['tasa_pct'], 100 - row['tasa_pct']],
        labels=['Abandono', 'Permanencia'],
        marker_colors=[COLOR_ABANDONO, COLOR_PERMANENCIA],
        hole=0.5, textinfo='percent', textfont_size=14,
        hovertemplate=f'<b>{row["label"]}</b><br>%{{label}}: %{{value:.1f}}%<br>n={int(row["n_total"]):,}<extra></extra>',
        name=row['label'],
    ), row=1, col=i+1)
fig.update_layout(title='Abandono según Beca', height=320, template='plotly_white',
                  showlegend=True, legend=dict(orientation='h', y=-0.05, x=0.5, xanchor='center'),
                  margin=dict(t=60, b=40))
graficos_html['beca'] = fig.to_html(full_html=False, include_plotlyjs=False, config=plotly_cfg)

# ---------------------------------------------------------------
# 4. VIA ACCESO — Barras horizontales
# ---------------------------------------------------------------
via_g2 = tasa_por_grupo('via_acceso', min_n=30).sort_values('tasa_pct', ascending=True)

fig = go.Figure(go.Bar(
    y=via_g2['via_acceso'], x=via_g2['tasa_pct'], orientation='h',
    marker_color=['#e53e3e' if t >= tasa_global else '#3182ce' for t in via_g2['tasa_pct']],
    text=[f"{t}%" for t in via_g2['tasa_pct']],
    textposition='outside', textfont_size=10,
    hovertemplate='<b>%{y}</b><br>Tasa: %{x:.1f}%<br>Abandonos: %{customdata[0]:,}<br>Total: %{customdata[1]:,}<extra></extra>',
    customdata=list(zip(via_g2['n_abandono'], via_g2['n_total']))
))
fig.add_vline(x=tasa_global, line_dash='dash', line_color='#e53e3e', line_width=2,
              annotation_text=f'Media: {tasa_global:.1f}%', annotation_position='top',
              annotation_font=dict(size=11, color='#e53e3e'))
fig.update_layout(title='Abandono por Via de Acceso', xaxis_title='Tasa de abandono (%)',
                  xaxis=dict(range=[0, via_g2['tasa_pct'].max() * 1.15]),
                  height=max(380, len(via_g2)*42), template='plotly_white')
graficos_html['via_acceso'] = fig.to_html(full_html=False, include_plotlyjs=False, config=plotly_cfg)



# ---------------------------------------------------------------
# 5. UNIVERSIDAD — Barras horizontales apiladas (como rama)
# ---------------------------------------------------------------
uni_g = tasa_por_grupo('universidad_origen', min_n=10).sort_values('tasa_pct', ascending=True)
uni_g['perm_pct'] = (100 - uni_g['tasa_pct']).round(1)

fig = go.Figure()
fig.add_trace(go.Bar(
    y=uni_g['universidad_origen'], x=uni_g['tasa_pct'], orientation='h',
    name='Abandono', marker_color=COLOR_ABANDONO,
    text=[f"{t}%" for t in uni_g['tasa_pct']], textposition='inside',
    textfont=dict(color='white', size=12),
    hovertemplate='<b>%{y}</b><br>Abandono: %{x:.1f}%<br>n=%{customdata[0]:,} de %{customdata[1]:,}<extra></extra>',
    customdata=list(zip(uni_g['n_abandono'], uni_g['n_total']))
))
fig.add_trace(go.Bar(
    y=uni_g['universidad_origen'], x=uni_g['perm_pct'], orientation='h',
    name='Permanencia', marker_color=COLOR_PERMANENCIA, opacity=0.6,
    text=[f"{t}%" for t in uni_g['perm_pct']], textposition='inside',
    textfont=dict(color='white', size=11),
    hovertemplate='<b>%{y}</b><br>Permanencia: %{x:.1f}%<extra></extra>',
))
fig.update_layout(barmode='stack', title='Abandono por Universidad de Origen',
                  xaxis_title='Porcentaje (%)', height=400, template='plotly_white',
                  legend=dict(orientation='h', x=1, y=1.02, xanchor='right', yanchor='bottom', bgcolor='rgba(255,255,255,0.85)', bordercolor='#e2e8f0', borderwidth=1, font=dict(size=11)),
                  margin=dict(b=40))
graficos_html['universidad'] = fig.to_html(full_html=False, include_plotlyjs=False, config=plotly_cfg)

# ---------------------------------------------------------------
# 6. SITUACION LABORAL — Etiquetas cortas + colorscale
# ---------------------------------------------------------------
sit_g2 = tasa_por_grupo('situacion_laboral', min_n=20).sort_values('tasa_pct', ascending=True)

# Diccionario para acortar nombres
SIT_CORTO = {
    'Artesanos y trabajadores calificados de las industrias manufactureras, la construcción y la minería, excepto los operadores de instalaciones y maquinaria.': 'Artesanos/Industria/Construccion',
    'Trabajadores de los servicios de restauración, personales, protección y vendedores de los comercios': 'Servicios/Restauracion/Comercio',
    'Operadores de instalaciones y maquinaria y montadores': 'Operadores/Maquinaria',
    'Técnicos y Profesionales científicos e intelectuales': 'Profesionales cientificos',
    'Técnicos y Profesionales de apoyo': 'Profesionales de apoyo',
    'Empleados de tipo administrativo': 'Administrativos',
    'Trabajadores calificados en la agricultura y la pesca': 'Agricultura/Pesca',
    'Dirección de Empresas y de las Administraciones Públicas': 'Directivos empresa/AAPP',
    'Trabajadores no calificados': 'Trabajadores no cualificados',
    'Inactivo o desempleado': 'Inactivo/desempleado',
    'Fuerzas Armadas': 'Fuerzas Armadas',
}
sit_g2['label'] = sit_g2['situacion_laboral'].map(SIT_CORTO).fillna(sit_g2['situacion_laboral'])

fig = go.Figure(go.Bar(
    y=sit_g2['label'], x=sit_g2['tasa_pct'], orientation='h',
    marker=dict(
        color=sit_g2['tasa_pct'],
        colorscale=[[0, '#38a169'], [0.5, '#ed8936'], [1, '#e53e3e']],
        showscale=False
    ),
    text=[f"<b>{t}%</b> (n={na:,})" for t, na in zip(sit_g2['tasa_pct'], sit_g2['n_abandono'])],
    textposition='outside', textfont_size=11,
    cliponaxis=False,
    hovertemplate='<b>%{y}</b><br>Tasa: %{x:.1f}%<br>Abandonos: %{customdata[0]:,}<br>Total: %{customdata[1]:,}<extra></extra>',
    customdata=list(zip(sit_g2['n_abandono'], sit_g2['n_total']))
))
fig.add_vline(x=tasa_global, line_dash='dash', line_color='#718096', line_width=2,
              annotation_text=f'Global: {tasa_global:.1f}%', annotation_font_size=11)
fig.update_layout(title='Abandono por Situación Laboral', xaxis_title='Tasa de abandono (%)',
                  height=max(380, len(sit_g2)*42), template='plotly_white',
                  margin=dict(r=100))
graficos_html['sit_laboral'] = fig.to_html(full_html=False, include_plotlyjs=False, config=plotly_cfg)

# ---------------------------------------------------------------
# 7. TITULACIONES — con desplegable (orden forzado en eje Y)
# ---------------------------------------------------------------
tit_all = tasa_por_grupo('titulacion', min_n=30)
# Abreviar nombres conservando "(Plan)" para distinguir duplicados
def acortar_tit(s):
    s = s.replace('Grado en ', '')
    s = s.replace('Ingeniería ', 'Ing. ').replace('Ingenieria ', 'Ing. ')
    s = s.replace('Administración de Empresas', 'ADE')
    s = s.replace('Administración y Dirección de Empresas', 'ADE')
    s = s.replace('Maestro en Educación ', 'Maestro Ed. ')
    s = s.replace('Relaciones Laborales y Recursos Humanos', 'RRLL y RRHH')
    s = s.replace('Publicidad y Relaciones Públicas', 'Publicidad y RRPP')
    s = s.replace('Comunicación Audiovisual', 'Com. Audiovisual')
    s = s.replace('Criminología y Seguridad', 'Criminología')
    s = s.replace('Traducción e Interpretación', 'Traducción')
    s = s.replace('Agroalimentaria y del Medio Rural', 'Agroalimentaria')
    s = s.replace('Diseño Industrial y Desarrollo de Productos', 'Diseño Industrial')
    s = s.replace('Diseño y Desarrollo de Videojuegos', 'Videojuegos')
    s = s.replace('Tecnologías Industriales', 'Tec. Industriales')
    s = s.replace('Gestión y Administración Pública', 'Gestión Pública')
    s = s.replace('Matemática Computacional', 'Mat. Computacional')
    s = s.replace('Finanzas y Contabilidad', 'Finanzas y Cont.')
    s = s.replace('Historia y Patrimonio', 'Historia')
    s = s.replace('Humanidades: Estudios Interculturales', 'Humanidades')
    # Mantener "(Plan XXXX)" pero acortar a "(20XX)"
    import re
    plan = re.search(r'\(Plan (\d{4})\)', s)
    if plan:
        s = re.sub(r' \(Plan \d{4}\)', '', s)
        s = s.strip()[:25] + f' ({plan.group(1)})'
    else:
        s = s[:30]
    return s

tit_all['tit_corto'] = tit_all['titulacion'].apply(acortar_tit)

tit_sorted = tit_all.sort_values('tasa_pct', ascending=True)
top10 = tit_all.head(10).sort_values('tasa_pct', ascending=True)
bot10 = tit_all.tail(10).sort_values('tasa_pct', ascending=True)

# Orden forzado para cada vista
orden_todas = tit_sorted['tit_corto'].tolist()
orden_top = top10['tit_corto'].tolist()
orden_bot = bot10['tit_corto'].tolist()

fig = go.Figure()

for data, name, visible in [(tit_sorted, 'Todas', True), (top10, 'Top 10', False), (bot10, 'Bottom 10', False)]:
    fig.add_trace(go.Bar(
        y=data['tit_corto'], x=data['tasa_pct'], orientation='h',
        marker=dict(color=data['tasa_pct'],
                    colorscale=[[0, '#38a169'], [0.5, '#ed8936'], [1, '#e53e3e']], showscale=False),
        text=[f"<b>{t}%</b> (n={na})" for t, na in zip(data['tasa_pct'], data['n_abandono'])],
        textposition='outside',
        cliponaxis=False, textfont_size=10,
        hovertemplate='<b>%{y}</b><br>Tasa: %{x:.1f}%<br>Abandonos: %{customdata[0]:,}<br>Total: %{customdata[1]:,}<extra></extra>',
        customdata=list(zip(data['n_abandono'], data['n_total'])),
        visible=visible, name=name
    ))

fig.add_vline(x=tasa_global, line_dash='dash', line_color='#718096', line_width=2,
              annotation_text=f'Global: {tasa_global:.1f}%')
fig.update_layout(
    title='Abandono por Titulación', xaxis_title='Tasa de abandono (%)',
    height=max(550, len(tit_all)*22), template='plotly_white',
    margin=dict(t=60, r=100),
    yaxis=dict(categoryorder='array', categoryarray=orden_todas),
    updatemenus=[{
        'buttons': [
            {'label': '  Todas las titulaciones  ', 'method': 'update',
             'args': [{'visible': [True, False, False]},
                      {'height': max(550, len(tit_all)*22),
                       'yaxis.categoryorder': 'array', 'yaxis.categoryarray': orden_todas}]},
            {'label': '  Top 10 mayor abandono  ', 'method': 'update',
             'args': [{'visible': [False, True, False]},
                      {'height': 450,
                       'yaxis.categoryorder': 'array', 'yaxis.categoryarray': orden_top}]},
            {'label': '  Top 10 menor abandono  ', 'method': 'update',
             'args': [{'visible': [False, False, True]},
                      {'height': 450,
                       'yaxis.categoryorder': 'array', 'yaxis.categoryarray': orden_bot}]},
        ],
        'direction': 'down', 'showactive': True,
        'x': 0.5, 'xanchor': 'left', 'y': 1.08, 'yanchor': 'top',
        'bgcolor': '#edf2f7', 'bordercolor': '#3182ce', 'font': {'size': 12}
    }]
)
graficos_html['titulacion'] = fig.to_html(full_html=False, include_plotlyjs=False, config=plotly_cfg)

print(f'Generados {len(graficos_html)} graficos Plotly')

Generados 7 graficos Plotly


In [5]:
# ============================================================================
# CELDA 4: SUNBURST, RANKING Y CONTRIBUCION
# ============================================================================

# --- Barras divergentes: titulaciones vs media global ---
tit_div = tasa_por_grupo('titulacion', min_n=30).copy()
# Acortar nombres de titulaciones
tit_div['tit_corto'] = tit_div['titulacion'].apply(acortar_tit)
tit_div['diff'] = (tit_div['tasa_pct'] - tasa_global).round(1)
tit_div = tit_div.sort_values('diff', ascending=True)

fig = go.Figure(go.Bar(
    y=tit_div['tit_corto'], x=tit_div['diff'], orientation='h',
    marker=dict(
        color=[COLOR_ABANDONO if d > 0 else COLOR_PERMANENCIA for d in tit_div['diff']],
        opacity=0.85
    ),
    text=[f"+{d}" if d > 0 else str(d) for d in tit_div['diff']],
    textposition='outside', textfont_size=10,
    hovertemplate='<b>%{y}</b><br>Tasa: %{customdata[0]}%<br>Diferencia vs global: %{x:+.1f} pp<br>n=%{customdata[1]:,}<extra></extra>',
    customdata=list(zip(tit_div['tasa_pct'], tit_div['n_total']))
))
fig.add_vline(x=0, line_color='#2d3748', line_width=2)
fig.add_annotation(x=max(tit_div['diff'])*0.95, y=len(tit_div)-1,
                   text=f'Por encima de la media ({tasa_global:.1f}%)',
                   showarrow=False, font=dict(size=10, color='rgba(229,62,62,0.6)'),
                   xanchor='right', yanchor='bottom')
# Anotación 'Por debajo' eliminada por limpieza visual
fig.update_layout(
    title='Desviación respecto a la Tasa Global de Abandono',
    xaxis_title='Diferencia en puntos porcentuales vs media global',
    height=max(600, len(tit_div)*22), template='plotly_white',
    yaxis=dict(showgrid=True, gridcolor='rgba(0,0,0,0.04)', gridwidth=1),
    xaxis=dict(zeroline=True, zerolinecolor='#4a5568', zerolinewidth=2,
               title='Diferencia en puntos porcentuales vs media global'),
    margin=dict(r=60))
graficos_html['divergente'] = fig.to_html(full_html=False, include_plotlyjs=False, config=plotly_cfg)

# --- Ranking: rama x via_acceso ---
ranking = (df.groupby(['rama', 'via_acceso'])['abandono']
           .agg(['mean', 'sum', 'count']).reset_index())
ranking.columns = ['rama', 'via_acceso', 'tasa', 'n_abandono', 'n_total']
ranking = ranking[ranking['n_total'] >= 20]
ranking['tasa_pct'] = (ranking['tasa'] * 100).round(1)
ranking['n_abandono'] = ranking['n_abandono'].astype(int)
ranking['n_total'] = ranking['n_total'].astype(int)
ranking = ranking.sort_values('tasa', ascending=True).tail(10)
ranking['label'] = ranking['rama'].map(DICCIONARIO_RAMAS).str[:20] + ' + ' + ranking['via_acceso'].str[:30]

fig = go.Figure(go.Bar(
    y=ranking['label'], x=ranking['tasa_pct'], orientation='h',
    marker=dict(color=ranking['tasa_pct'], colorscale=[[0, '#38a169'], [0.5, '#ed8936'], [1, '#e53e3e']], showscale=False),
    text=[f"{t}% ({na}/{nt})" for t, na, nt in zip(ranking['tasa_pct'], ranking['n_abandono'], ranking['n_total'])],
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Tasa: %{x:.1f}%<br>Abandonos: %{customdata[0]:,}<br>Total: %{customdata[1]:,}<extra></extra>',
    customdata=list(zip(ranking['n_abandono'], ranking['n_total']))
))
fig.add_vline(x=tasa_global, line_dash='dash', line_color='gray',
              annotation_text=f'Global: {tasa_global:.1f}%')
fig.update_layout(title='Top 10 Combinaciónes Rama x Via Acceso con Mayor Abandono',
                  xaxis_title='Tasa de abandono (%)', height=450, template='plotly_white')
graficos_html['ranking'] = fig.to_html(full_html=False, include_plotlyjs=False, config=plotly_cfg)

# --- Contribución al abandono ---
variables_contrib = ['rama', 'sexo', 'via_acceso', 'n_anios_beca']
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[f'Por {etiq(v)}' for v in variables_contrib])

for i, col in enumerate(variables_contrib):
    row, colp = (i // 2) + 1, (i % 2) + 1
    grupo = df[df['abandono']==1].groupby(col).size().reset_index(name='n')
    total_ab = grupo['n'].sum()
    grupo['pct'] = (grupo['n'] / total_ab * 100).round(1)
    grupo = grupo.sort_values('pct', ascending=True)
    labels = grupo[col].tolist()
    if col == 'rama':
        labels = [DICCIONARIO_RAMAS.get(l, l) for l in labels]
    elif col == 'n_anios_beca':
        labels = ['Sin beca' if l == 0 else f'{l} año(s) beca' for l in labels]
    else:
        labels = [str(l)[:25] for l in labels]

    fig.add_trace(go.Bar(
        y=labels, x=grupo['pct'], orientation='h',
        marker_color=COLOR_ABANDONO,
        text=[f'{p:.1f}%' for p in grupo['pct']], textposition='outside',
        hovertemplate='<b>%{y}</b><br>%{x:.1f}% del total de abandonos<br>n=%{customdata:,}<extra></extra>',
        customdata=grupo['n'],
        showlegend=False
    ), row=row, col=colp)

fig.update_layout(title_text='Contribución al Total de Abandonos',
                  height=500, template='plotly_white')
graficos_html['contribucion'] = fig.to_html(full_html=False, include_plotlyjs=False, config=plotly_cfg)

print(f'Total graficos: {len(graficos_html)}')

Total graficos: 10


In [6]:
# ============================================================================
# CELDA 5: GENERAR HTML
# ============================================================================

nav_fases, nav_modulos = generar_html_navegacion_completa(
    fase_activa='fase4', modulo_activo='m02'
)


def generar_kpis_html_con_tooltip(kpis):
    html = '<div class="kpis" style="gap:8px;">'
    for kpi in kpis:
        tooltip = kpi.get('tooltip', '')
        title_attr = f' title="{tooltip}"' if tooltip else ''
        html += (
            f'<div class="kpi" style="cursor:help;padding:10px 8px;"{title_attr}>'
            f'<div class="value" style="color:{kpi["color"]};font-size:1.4em;">{kpi["valor"]}</div>'
            f'<div class="label" style="font-size:0.75em;margin-top:4px;">{kpi["titulo"]}</div>'
            f'</div>'
        )
    html += '</div>'
    return html

KPIS = [
    {'valor': f'{tasa_global:.1f}%', 'titulo': 'Abandono global',
     'tooltip': f'{fmt(n_abandonan)} de {fmt(n_filas)} estudiantes. Periodo 2010-2021, UJI.',
     'color': '#e53e3e'},
    {'valor': f'{rama_max_tasa}%', 'titulo': 'Ing. y Arquitectura',
     'tooltip': f'Rama con mayor abandono (n={fmt(int(rama_g.iloc[0]["n_total"]))}). La menor: {DICCIONARIO_RAMAS.get(rama_min, rama_min)} con {rama_min_tasa}%.',
     'color': '#3182ce'},
    {'valor': f'{sexo_max_tasa:.1f}% / {df[df["sexo"].str.lower().str.contains("mujer|woman|female", na=False)]["abandono"].mean()*100:.1f}%' if df["sexo"].dtype == "object" else f'{sexo_max_tasa:.1f}%',
     'titulo': 'Abandono por sexo',
     'tooltip': 'Diferencia entre grupos por sexo. Analizada en Fase 6 fairness.',
     'color': '#38a169'},
    {'valor': f'{tasa_sin_beca:.1f}% / {tasa_con_beca:.1f}%',
     'titulo': 'Sin beca / Con beca',
     'tooltip': f'El apoyo económico reduce el abandono {tasa_sin_beca - tasa_con_beca:.1f} pp.',
     'color': '#d69e2e'},
    {'valor': f'{tasa_edad_joven:.0f}% a {tasa_edad_mayor:.0f}%',
     'titulo': 'Edad: 18 vs 26-30',
     'tooltip': f'Gradiente continuo: a mayor edad de entrada, mayor abandono.',
     'color': '#805ad5'},
    {'valor': f'{fmt(n_filas)}', 'titulo': f'{n_titulaciones} grados · {n_ramas} ramas',
     'tooltip': f'{fmt(n_filas)} estudiantes, {n_titulaciones} titulaciones, {n_ramas} ramas. Periodo 2010-2021.',
     'color': '#4a5568'},
]

plotly_cdn = '<script src="https://cdn.plot.ly/plotly-3.4.0.min.js"></script>'


def g(key):
    return graficos_html[key]

s_rama = generar_seccion_html('Abandono por Rama de Conocimiento', g('rama'))

s_sexo_beca = generar_seccion_html('Abandono por Sexo y Beca',
    '<div style="display:grid;grid-template-columns:1fr 1fr;gap:10px;">'
    f'<div>{g("sexo")}</div><div>{g("beca")}</div></div>')

s_via = generar_seccion_html('Abandono por Via de Acceso', g('via_acceso'))
s_uni = generar_seccion_html('Abandono por Universidad de Origen', g('universidad'))

s_tit = generar_seccion_html('Abandono por Titulación',
    g('titulacion')
    + '<div style="background:#ebf8ff;padding:8px;border-radius:8px;border-left:4px solid #3182ce;margin-top:8px;font-size:0.85em;">'
    '<strong>Usa el desplegable</strong> para ver: todas las titulaciones, top 10 mayor abandono o top 10 menor abandono.'
    '</div>')

s_sit = generar_seccion_html('Abandono por Situación Laboral', g('sit_laboral'))

s_sunburst = generar_seccion_html('Titulaciónes vs Media Global',
    g('divergente')
    + '<div style="background:#f0f4f8;padding:8px;border-radius:8px;border-left:4px solid #3182ce;margin-top:8px;font-size:0.85em;">'
    '<strong>Lectura:</strong> Barras rojas = titulaciones con tasa de abandono superior a la media global. ''Azules = por debajo. La longitud indica cuantos puntos porcentuales se desvian.'
    '</div>')

s_ranking = generar_seccion_html('Ranking de Riesgo: Rama x Via Acceso',
    g('ranking')
    + '<div style="background:#fff5f5;padding:8px;border-radius:8px;border-left:4px solid #e53e3e;margin-top:8px;font-size:0.85em;">'
    f'<strong>Lectura:</strong> Combinaciónes con mayor tasa de abandono. '
    f'Línea discontinua = tasa global ({tasa_global:.1f}%). Ratio = abandonos/total.'
    '</div>')

s_contrib = generar_seccion_html('Contribución al Total de Abandonos',
    g('contribucion')
    + '<div style="background:#f0f4f8;padding:8px;border-radius:8px;border-left:4px solid #3182ce;margin-top:8px;font-size:0.85em;">'
    '<strong>Nota:</strong> La tasa dice que % de un grupo abandona. '
    'La contribucion dice que % del total de abandonos proviene de ese grupo.'
    '</div>')

contenido = (
    plotly_cdn
    + generar_kpis_html_con_tooltip(KPIS)
    + s_rama
    + s_sexo_beca
    + s_via
    + s_uni
    + s_tit
    + s_sit
    + s_sunburst
    + s_ranking
    + s_contrib
)

html = render_base_html(
    titulo='M02: Análisis del Target',
    icono=chr(0x1f3af),
    subtitulo='Fase 4: EDA Final | TFM Abandono Universitario',
    nav_fases=nav_fases,
    nav_modulos=nav_modulos,
    contenido=contenido,
    notebook_nombre='f4_m02_target.ipynb',
    notebook_carpeta='fase4_eda',
)

ruta_html = RUTA_FASE4_HTML / 'm02_target.html'
guardar_html(html, ruta_html)
print(f'HTML generado: {ruta_html}')

✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m02_target.html
HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m02_target.html


In [7]:
# ============================================================================
# CELDA 6: RESUMEN
# ============================================================================

print('='*70)
print('F4-M02: TARGET - RESUMEN')
print('='*70)
print(f'Tasa global:     {tasa_global:.1f}%')
print(f'Graficos Plotly:  {len(graficos_html)}')
print(f'  Interactivos:  tooltips, desplegable titulaciones, sunburst navegable')
print(f'Variables:       rama, sexo, beca, via_acceso, universidad,')
print(f'                 titulacion, situacion_laboral')
print(f'Extras:          ranking rama x via, contribucion al abandono')
print(f'HTML:            {ruta_html}')
print()
print('Siguiente: f4_m03_bivariante_num.ipynb')

F4-M02: TARGET - RESUMEN
Tasa global:     29.2%
Graficos Plotly:  10
  Interactivos:  tooltips, desplegable titulaciones, sunburst navegable
Variables:       rama, sexo, beca, via_acceso, universidad,
                 titulacion, situacion_laboral
Extras:          ranking rama x via, contribucion al abandono
HTML:            C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase4\m02_target.html

Siguiente: f4_m03_bivariante_num.ipynb
